#### From Text to Diagnosis:
#### Evaluating GPT-Based Models for Classifying Depression Severity in Online Texts

# Script 2.1 - PROMPT ENGINEERING (Realtime)

Metrics and Prompts were updated on version 2.2

In [ ]:
# IMPORTS #

import os
import pandas as pd
import numpy as np
import json
import openai
from sklearn.metrics import confusion_matrix, mean_absolute_error, mean_squared_error

## Part 1. Logic

In [ ]:
# Set up the OpenAI API key.
# Enter the OpenAI API key when prompted. Do not store API keys in notebooks.
api_key = input('Enter your OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = api_key
openai.api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
# Data load.
#df = pd.read_csv(r"C:/Users/scotl/OneDrive/Desktop/data/data validation.csv", delimiter=',')
df = pd.read_csv(r"C:\Users\marti\Downloads\data validation.csv", delimiter=',')
#df = pd.read_csv(r"C:/Users/Main/Desktop/data/data validation.csv", delimiter=',')

# Param. to impose row limit to reduce spending; used ONLY for testing.
limit_rows = True

if limit_rows == True:
    df = df.head(10)

### Functions

In [ ]:
# Function to call the OpenAI API to generate (and return) text repsonse based on the input prompt.
def get_completion(prompt, model):
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


### Perhaps we should also pass the df in case we want to test different dfs? ###
def compare_depression_levels(prompt_type, model):
    results = []
    
    label_mapping = {"minimum": 0, "mild": 1, "moderate": 2, "severe": 3, 0: 0, 1: 1, 2: 2, 3: 3}
    
    for i in range(len(df)):
        # Pull a row text from the data and create a prompt, based on the selected template.
        text = df.at[i, 'text']
        prompt = construct_prompt(text, prompt_type)
        
        try:
            response = get_completion(prompt, model)
            print(f"Row {i} - Raw response:", response)
            response = response.strip().strip('```').strip('json').strip()
            response_json = json.loads(response)
            response_json = normalize_key_names(response_json)
            
            depression_level = response_json.get('depression_level', None)

            if depression_level is None:
                print(f"Row {i} - Invalid depression level: {depression_level}")
                continue

            # Check if depression_level is already an integer or convert it if it's a string.
            # If not one of the correct valeus, set to -1.
            if isinstance(depression_level, str):
                depression_level = label_mapping.get(depression_level.lower(), -1)
            elif isinstance(depression_level, int):
                depression_level = label_mapping.get(depression_level, -1)
            
            # If fallback value:
            if depression_level == -1:
                print(f"Row {i} - Invalid depression level: {depression_level}")
                continue
            
            # Compare predicted and actual level, to populate match col with True/False.
            actual_label = int(df.at[i, 'multi_label_number'])
            result = {
                "index": i,
                "predicted_depression_level": depression_level,
                "actual_depression_level": actual_label,
                "match": depression_level == actual_label
            }
            results.append(result)
        
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Error parsing response at row {i}: {e}")
            print(f"Response was: {response}")
            continue

    return results


def normalize_key_names(response_json):
    """
    Function to normalize the keys in the response JSON to ensure consistent key (and value, if text) naming.
    E.g. calculate the reponse regarless of whether the key is "depression_level" or "Depression Level".
    """
    ### Perhaps a better approach would be ignoring the key alltogether? ###
    normalized_response = {}
    for key in response_json:
        normalized_key = key.lower().replace(" ", "_")
        normalized_response[normalized_key] = response_json[key]
    return normalized_response


# Function to calculate and print metrics
def calculate_and_print_metrics(results_df):
    # Check if the DataFrame is not empty and contains the 'match' column
    if not results_df.empty and 'match' in results_df.columns:
        
        # Uncomment these to inspect the process:
        # COMMENT DURING PROPER RUN!
        print("Initial DataFrame:")
        print(results_df)
        
        # Remove the 'index' column if it exists
        if 'index' in results_df.columns:
            results_df = results_df.drop(columns=['index'])
        
        # Drop rows with NaN values in 'predicted_depression_level' or 'actual_depression_level'
        results_df = results_df.dropna(subset=['predicted_depression_level', 'actual_depression_level'])
        
        # Uncomment these to inspect the process:
        # COMMENT DURING PROPER RUN!
        print("After dropping NaNs:")
        print(results_df)

        # Ensure the columns are of the correct type and filter out non-numeric values.
        results_df = results_df[results_df['predicted_depression_level'].apply(lambda x: isinstance(x, int))]
        results_df['predicted_depression_level'] = results_df['predicted_depression_level'].astype(int)
        results_df['actual_depression_level'] = results_df['actual_depression_level'].astype(int)
        
        # Uncomment these to inspect the process:
        # COMMENT DURING PROPER RUN!
        print("After filtering non-numeric values:")
        print(results_df)

        # Calculate and print metrics.
        accuracy = results_df['match'].mean()
        print(f"Accuracy: {accuracy * 100:.2f}%")

        y_true = results_df['actual_depression_level']
        y_pred = results_df['predicted_depression_level']

        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        conf_matrix = confusion_matrix(y_true, y_pred)

        print(f"Mean Absolute Error (MAE): {mae}")
        print(f"Root Mean Squared Error (RMSE): {rmse}")
        print("Confusion Matrix:")
        print(conf_matrix)
    else:
        print("No valid results to display or 'match' column missing.")

### Execution

In [ ]:
model = "gpt-3.5-turbo-0125"

# Use function with different prompts 
for prompt in ['Basic Zero-Shot', 'Zero-Shot Scale', 'Zero-Shot Role and Context', 'Zero-Shot Mental', 'Few-shot Context and Role', 'Few-shot Context, Role and Rationale', 'Complex Instructions', 'Chain of thought with rationale', 'Chain of thought no rationale', 'Chain of thought with examples']:
    comparison_results = compare_depression_levels(prompt, model)
    results_df = pd.DataFrame(comparison_results)
    print(f"Metrics for {prompt}:")
    calculate_and_print_metrics(results_df)

## Part 2. Prompts

In [ ]:
# Note that you can change the name of the prompts (prompt1 etc.) to better reflect the methodology.
# Remember to also change the name in the function list above!

def construct_prompt(text, prompt_type):
    if prompt_type == 'Basic Zero-Shot':
        return f"""
            Question: What is the depression level of this user?

            Output Constraint: Only return the classification of \
            depression level (Minimum, Mild, Moderate, Severe).

            Text data: ```{text}```
        """
    elif prompt_type == 'Zero-Shot Scale':
        return f"""
            Question: What is the depression level of this user?

            Output Constraint: Only return the depression level \
            (Minimum, Mild, Moderate, Severe) in a JSON format, \
            with Minimum being no to minimal signs of depression, \
            Mild being mild signs of depression, \
            Moderate being moderate signs of depression, \
            and Severe being severely depressed.

            Text data: ```{text}```
        """
    elif prompt_type == 'Zero-Shot Role and Context':
        return f"""
            You are a mental health professional specializing in depression. \
            Your task is to analyze a reddit post from a patient and answer the following question.
            What is the depression level of this user?

            Output Constraint: Only return the depression level \
            (Minimum, Mild, Moderate, Severe) in a JSON format, \
            with Minimum being no to minimal signs of depression, \
            Mild being mild signs of depression, \
            Moderate being moderate signs of depression, \
            and Severe being severely depressed.

            Text: ```{text}```
        """
    elif prompt_type == 'Zero-Shot Mental':
        return f"""
            This person wrote this paragraph on social media. \
            As a psychologist, read the post on social media and answer the question.
            Which level is the person [depressed]?

            Output Constraint: Only return the depression level (minimum, mild, moderate, severe)!
            Text: ```{text}```
        """
    elif prompt_type == 'Few-shot Context and Role':
        return f"""
            Instruction: You will be given text data from users. \
            Assume the role of a psychologist and rank the depression level of the user \
            based on the text from the patients' Reddit posts. \
            The depression level is on a scale of Minimum to Severe, \
            where Minimum is no to minimal signs of depression, \
            Mild is mild signs of depression, \
            Moderate is moderate signs of depression, \
            and Severe is very depressed.
            
            Examples:
            [Wtf. I did this to myself. I knew I needed to study and I just kept putting it off \
            and putting it off. I knew I didnâ€™t do well but I didnâ€™t think I bombed it so fucking hard. \
            Iâ€™m going to have to lie to everyone about it.] ["Minimum"]
            
            [I don't know if I'm able to do that, since I didn't tell the doctor the truth to begin with. \
            I will say that my doctor very clearly didn't believe me at my follow up appointment, \
            but my bf was in the waiting room. So I didn't admit to anything. \
            I was teary eyed, however. From anyone's experience- is it worth it to go to the police?] [Mild]

            [I drove the 10 minutes home with my fiance on the phone helping me count my breathing. \
            I got home and collapsed. I couldn't breathe. My neck and chest hurt. I was hysterical.] \
            ["Moderate"]

            [It feels great, but I'm worried. I tried to kill myself one year and one month ago. \
            I am prone to depression. I am worried about crashing too hard, and I have kids to care for. \
            Has anyone else experienced anything like this?] ["Severe"]

            Text: ```{text}```

            Format the output in JSON format. For example:
            {{
                "depression_level": "Mild"
            }}
        """
    elif prompt_type == 'Few-shot Context, Role and Rationale':
        return f"""
            Instruction: You will be given text data from users' Reddit posts. \
            Assume the role of a psychologist and rank the depression level \
            of the user based on the text. \
            The depression level is on a scale of Minimum to Severe, \
            where Minimum is no to minimal signs of depression, \
            Mild is mild signs of depression, \
            Moderate is moderate signs of depression, \
            and Severe is very depressed. \
            Provide reasoning for the depression level detected in the text and \
            base your classification on it.

            Examples:
            [Wtf. I did this to myself. I knew I needed to study and I just kept putting it off \
            and putting it off. I knew I didnâ€™t do well but I didnâ€™t think I bombed it so fucking hard. \
            Iâ€™m going to have to lie to everyone about it.] ["Minimum"]
            
            [I don't know if I'm able to do that, since I didn't tell the doctor the truth to begin with. \
            I will say that my doctor very clearly didn't believe me at my follow up appointment, \
            but my bf was in the waiting room. So I didn't admit to anything. \
            I was teary eyed, however. From anyone's experience- is it worth it to go to the police?] [Mild]

            [I drove the 10 minutes home with my fiance on the phone helping me count my breathing. \
            I got home and collapsed. I couldn't breathe. My neck and chest hurt. I was hysterical.] \
            ["Moderate"]

            [It feels great, but I'm worried. I tried to kill myself one year and one month ago. \
            I am prone to depression. I am worried about crashing too hard, and I have kids to care for. \
            Has anyone else experienced anything like this?] ["Severe"]
            
            Text: ```{text}```

            Format the output in JSON format, for example:
            {{
                "rationale": "The text is about a happy moment in my life.",
                "depression_level": "Mild"
            }}
        """
    elif prompt_type == 'Complex Instructions':
        return f"""
            Instruction: You are a psychologist analyzing text data from users to determine their \
            depression level. Rank the depression level of the user based on the text provided. \
            The depression level is on a scale of Minimum to Severe, where:
            Minimum = No to minimal signs of depression
            Mild = Mild signs of depression
            Moderate = Moderate signs of depression
            Severe = Severe signs of depression

            Your response should include:
            1. A brief rationale explaining why you assigned this depression level (no more than 20 words).
            2. The depression level in JSON format.

            Question: What is the depression level of this user?

            Output Constraint: Only return the classification of depression level \
            (Minimum, Mild, Moderate, Severe) and the rationale in a JSON format.

            Text data: ```{text}```

            Example Output:
            {{
                "rationale": "The user expresses feelings of severe hopelessness and isolation.",
                "depression_level": "Severe"
            }}
        """
    elif prompt_type == 'Chain of thought with rationale':
        return f"""
            Instruction: You are a psychologist analyzing text data from users' reddit posts to determine \
            their depression level. Rank the depression level of the user based on the text provided. \
            The depression level is on a scale of Minilal to Severe, where:
            Minimal = No to minimal signs of depression
            Mild = Mild signs of depression
            Moderate = Moderate signs of depression
            Severe = Severe signs of depression

            Use the following chain of thought to analyze the text:

            1. Identify any explicit statements related to feelings of \
            sadness, hopelessness, failure, or unhappiness.
            2. Assess the frequency and intensity of these feelings based on the user's descriptions.
            3. Consider any additional context that might indicate a higher or lower depression level.
            4. Based on your assessment, assign a depression level from 0 to 3.
            5. Provide a brief rationale explaining why you assigned this depression level \
            (no more than 20 words).

            Your response should include:
            1. A brief rationale explaining why you assigned this depression level.
            2. The depression level in JSON format.

            Question: What is the depression level of this user?

            Output Constraint: Only return the classification of depression level (Minimum, Mild, Moderate, Severe) and \
            the rationale in a JSON format.

            Text data: ```{text}```

            Example Output:
            {{
                "rationale": "The user expresses feelings of severe hopelessness and isolation.",
                "depression_level": 3
            }}
        """
    elif prompt_type == 'Chain of thought no rationale':
        return f"""
            Instruction: You are a psychologist analyzing text data from users' reddit posts to \
            determine their depression level. Rank the depression level of the user based on the \
            text provided. The depression level is on a scale of Minimal to Severe, where:
            Minimal = No to minimal signs of depression
            Mild = Mild signs of depression
            Moderate = Moderate signs of depression
            Severe = Severe signs of depression

            Use the following chain of thought to analyze the text:

            1. Identify any explicit statements related to feelings of \
            sadness, hopelessness, failure, or unhappiness.
            2. Assess the frequency and intensity of these feelings based on the user's descriptions.
            3. Consider any additional context that might indicate a higher or lower depression level.
            4. Based on your assessment, assign a depression level from 0 to 3.

            Question: What is the depression level of this user?

            Output Constraint: Only return the classification of depression level (Minimum, Mild, Moderate, Severe) \
            in a JSON format.

            Text data: ```{text}```

            Example Output:
            {{
                "depression_level": 3
            }}
        """
    elif prompt_type == 'Chain of thought with examples':
        return f"""
            Instruction: You are a psychologist analyzing text data from users' reddit posts to \
            determine their depression level. Rank the depression level of the user based on the \
            text provided. The depression level is on a scale of Minimal to Severe, where:
            Minimal = No to minimal signs of depression
            Mild = Mild signs of depression
            Moderate = Moderate signs of depression
            Severe = Severe signs of depression

            Use the following chain of thought to analyze the text:

            1. Identify any explicit statements related to feelings of \
            sadness, hopelessness, failure, or unhappiness.
            2. Assess the frequency and intensity of these feelings based on the user's descriptions.
            3. Consider any additional context that might indicate a higher or lower depression level.
            4. Based on your assessment, assign a depression level from 0 to 3.

            These are examples of your previous work:
            
            [Wtf. I did this to myself. I knew I needed to study and I just kept putting it off \
            and putting it off. I knew I didnâ€™t do well but I didnâ€™t think I bombed it so fucking hard. \
            Iâ€™m going to have to lie to everyone about it.] ["Minimum"]
            
            [I don't know if I'm able to do that, since I didn't tell the doctor the truth to begin with. \
            I will say that my doctor very clearly didn't believe me at my follow up appointment, but my \
            bf was in the waiting room. So I didn't admit to anything. I was teary eyed, however. \
            From anyone's experience- is it worth it to go to the police?] [Mild]

            [I drove the 10 minutes home with my fiance on the phone helping me count my breathing. \
            I got home and collapsed. I couldn't breathe. My neck and chest hurt. I was hysterical.] \
            ["Moderate"]

            [It feels great, but I'm worried. I tried to kill myself one year and one month ago. \
            I am prone to depression. I am worried about crashing too hard, and I have kids to care for. \
            Has anyone else experienced anything like this?] ["Severe"]

            Question: What is the depression level of this user?

            Output Constraint: Only return the classification of depression level (Minimum, Mild, Moderate, Severe) \
            in a JSON format.

            Text data: ```{text}```

            Example Output:
            {{
                "depression_level": 3
            }}
        """